In [ ]:
# -*- coding: utf-8 -*-
import os
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
import importlib.util

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
import torchbnn as bnn
from torch.utils.data import TensorDataset, DataLoader

# =========================================================
# Find repo root robustly whether execution starts in repo root or in code/
# =========================================================
HERE = Path.cwd()

if (HERE / "data").exists() and (HERE / "results").exists():
    REPO_ROOT = HERE
elif (HERE.parent / "data").exists() and (HERE.parent / "results").exists():
    REPO_ROOT = HERE.parent
else:
    raise FileNotFoundError(
        f"Could not locate repo root from working directory: {HERE}"
    )

print("Working directory:", HERE)
print("Resolved repo root:", REPO_ROOT)

# =========================================================
# Device
# =========================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# =========================================================
# Config
# =========================================================
DATA_PATH = REPO_ROOT / "data"
OUT_CSV = REPO_ROOT / "results" / "intermediate" / "bnn_table4.csv"
os.makedirs(REPO_ROOT / "results" / "intermediate", exist_ok=True)

TURBINES = list(range(1, 67))
TESTSET  = list(range(38,45)) 
TRAINSET = [i for i in TURBINES if i not in TESTSET]

FEATURES = ["wind_speed", "temperature"]
TARGET   = "power"


# =========================================================
# Terrain data
# =========================================================
terrain_path = DATA_PATH / "weightedTerrainData.csv"
terrain_df = pd.read_csv(terrain_path)

# Keep numeric columns
terrain_num = terrain_df.select_dtypes(include=[np.number]).copy()

# If first numeric column is just a turbine id/index, drop it.
# Adjust if needed after checking terrain_df.head().
if terrain_num.shape[1] > 3:
    terrain_num = terrain_num.iloc[:, 1:]

# Scale terrain once globally
terrain_scaler = StandardScaler()
terrain_scaled = terrain_scaler.fit_transform(terrain_num.to_numpy(np.float32))

# Map turbine id -> terrain vector
terrain_map = {
    tid: terrain_scaled[tid - 1]
    for tid in TURBINES
}


def build_xy_with_terrain(df, tid, feature_cols, target_col):
    df = df[feature_cols + [target_col]].dropna().copy()

    X_dyn = df[feature_cols].to_numpy(np.float32)
    y = df[target_col].to_numpy(np.float32).reshape(-1)

    terr = terrain_map[tid].astype(np.float32)
    terr_mat = np.repeat(terr.reshape(1, -1), repeats=len(df), axis=0)

    X = np.column_stack([X_dyn, terr_mat]).astype(np.float32)
    return X, y
# =========================================================
# BNN hyperparameters
# same BNN style as before
# =========================================================
SEED = 2048
BATCH_SIZE = 2048
EPOCHS = 100
LR = 0.001

H = 128
PRIOR_MU = 0.0
PRIOR_SIGMA = 0.1
KL_WEIGHT = 0.01
MC_DRAWS = 40
PRED_BATCH_SIZE = 8192

# =========================================================
# Seed
# =========================================================
def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

# =========================================================
# Helpers
# =========================================================
def read_csv_safe(path):
    path = Path(path)
    try:
        if path.exists():
            return pd.read_csv(path)
    except Exception:
        pass
    return None

def make_model(d_in):
    model = nn.Sequential(
        bnn.BayesLinear(
            prior_mu=PRIOR_MU,
            prior_sigma=PRIOR_SIGMA,
            in_features=d_in,
            out_features=H
        ),
        nn.ReLU(),
        bnn.BayesLinear(
            prior_mu=PRIOR_MU,
            prior_sigma=PRIOR_SIGMA,
            in_features=H,
            out_features=H
        ),
        nn.ReLU(),
        bnn.BayesLinear(
            prior_mu=PRIOR_MU,
            prior_sigma=PRIOR_SIGMA,
            in_features=H,
            out_features=1
        ),
    ).to(DEVICE)
    return model

mse_loss = nn.MSELoss()
kl_loss = bnn.BKLLoss(reduction="mean", last_layer_only=False)

def train_bnn(model, X_train, y_train):
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

    loader = DataLoader(
        TensorDataset(X_tensor, y_tensor),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        drop_last=False
    )

    optimizer = optim.Adam(model.parameters(), lr=LR)

    model.train()
    for epoch in range(EPOCHS):
        epoch_mse = 0.0
        epoch_kl = 0.0
        n_batches = 0

        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            pred = model(xb)
            mse = mse_loss(pred, yb)
            kl = kl_loss(model)
            loss = mse + KL_WEIGHT * kl

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            epoch_mse += mse.item()
            epoch_kl += kl.item()
            n_batches += 1

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"MSE: {epoch_mse / max(n_batches, 1):.4f} | "
            f"KL: {epoch_kl / max(n_batches, 1):.4f}"
        )

    return model

@torch.no_grad()
def predict_mc(model, X_test, mc_draws=MC_DRAWS, pred_batch_size=PRED_BATCH_SIZE):
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32)

    n = X_tensor.shape[0]
    draws = []

    for _ in range(mc_draws):
        preds = []
        for start in range(0, n, pred_batch_size):
            xb = X_tensor[start:start + pred_batch_size].to(DEVICE)
            pred = model(xb).cpu().numpy().reshape(-1)
            preds.append(pred)
        draws.append(np.concatenate(preds))

    draws = np.stack(draws, axis=0)   # (mc_draws, n_test)
    pred_mean = draws.mean(axis=0)
    pred_sd = draws.std(axis=0)

    return pred_mean, pred_sd

# =========================================================
# Build ONE training set from all 2017 data in TRAINSET
# =========================================================
X_train_parts = []
y_train_parts = []

for j in TRAINSET:
    tr = read_csv_safe(DATA_PATH / f"Turbine{j}_2017.csv")
    if tr is None or not set(FEATURES + [TARGET]).issubset(tr.columns):
        continue

    Xj, yj = build_xy_with_terrain(tr, j, FEATURES, TARGET)
    if len(yj) == 0:
        continue

    X_train_parts.append(Xj)
    y_train_parts.append(yj)

if len(X_train_parts) == 0:
    raise ValueError("No valid training data found in TRAINSET.")

X_train = np.vstack(X_train_parts).astype(np.float32)
y_train = np.concatenate(y_train_parts).astype(np.float32)
# ---- Scale X and y ----
x_scaler = StandardScaler()
X_train_s = X_train.copy()
X_train_s[:, :len(FEATURES)] = x_scaler.fit_transform(X_train[:, :len(FEATURES)]).astype(np.float32)

y_mean = float(np.mean(y_train))
y_std = float(np.std(y_train))
if (not np.isfinite(y_std)) or (y_std < 1e-8):
    y_std = 1.0

y_train_s = ((y_train - y_mean) / y_std).astype(np.float32)

# =========================================================
# Train model once
# =========================================================
t1 = time.time()
model = make_model(d_in=X_train_s.shape[1])
model = train_bnn(model, X_train_s, y_train_s)
t2 = time.time()
train_time = round(t2 - t1, 2)

# =========================================================
# Predict on each turbine in TESTSET (2017 only)
# =========================================================
rows = []

for i in TESTSET:
    print(f"\n=== Evaluating Turbine {i} on 2017 ===")

    test17 = read_csv_safe(DATA_PATH / f"Turbine{i}_2017.csv")
    if test17 is None or not set(FEATURES + [TARGET]).issubset(test17.columns):
        print(f"Skipping Turbine {i} (missing 2017 data or columns)")
        continue

    test17 = test17[FEATURES + [TARGET]].dropna()
    if test17.empty:
        print(f"Skipping Turbine {i} (empty after dropna)")
        continue

    X_te17, y_te17 = build_xy_with_terrain(test17, i, FEATURES, TARGET)

    X_te17_s = X_te17.copy()
    X_te17_s[:, :len(FEATURES)] = x_scaler.transform(X_te17[:, :len(FEATURES)]).astype(np.float32)

    p1 = time.time()
    pred17_s, pred17_sd_s = predict_mc(model, X_te17_s, mc_draws=MC_DRAWS)
    p2 = time.time()

    pred17 = pred17_s * y_std + y_mean
    pred17_sd = pred17_sd_s * y_std

    pred17 = np.clip(pred17, 0.0, None)

    pred17_time = round(p2 - p1, 2)
    rmse17 = float(np.sqrt(mean_squared_error(y_te17, pred17)))

    rows.append({
        "Turbine": i,
        "Method": "BNN",
        "RMSE_2017": rmse17,
        "Runtime_train": train_time,
        "Runtime_pred17": pred17_time
    })

    print(pd.DataFrame([rows[-1]]))

# =========================================================
# Save
# =========================================================
results = pd.DataFrame(
    rows,
    columns=["Turbine", "Method", "RMSE_2017", "Runtime_train", "Runtime_pred17"]
)

results.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
print(results.head())

# =========================================================
# Update final results.csv : Table 4, BNN row
# =========================================================
upd_path = REPO_ROOT / "code" / "update_final_results.py"
spec = importlib.util.spec_from_file_location("update_final_results", upd_path)
upd_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(upd_mod)

update_final_results = upd_mod.update_final_results

bnn_table4_rmse = results["RMSE_2017"].mean() if not results.empty else np.nan

update_final_results(method="Bayesian NN", table_id="Table 4", rmse=bnn_table4_rmse, version="xs")

# -------------------------------------------------
# BNN(x): retrain WITHOUT terrain features
# -------------------------------------------------
X_train_x_parts = []
y_train_x_parts = []
for j in TRAINSET:
    tr = read_csv_safe(DATA_PATH / f"Turbine{j}_2017.csv")
    if tr is None or not set(FEATURES+[TARGET]).issubset(tr.columns): continue
    tr = tr[FEATURES+[TARGET]].dropna()
    if tr.empty: continue
    X_train_x_parts.append(tr[FEATURES].to_numpy(np.float32))
    y_train_x_parts.append(tr[TARGET].to_numpy(np.float32))

X_train_x = np.vstack(X_train_x_parts).astype(np.float32)
y_train_x = np.concatenate(y_train_x_parts).astype(np.float32)

x_scaler_x = StandardScaler()
X_train_x_s = x_scaler_x.fit_transform(X_train_x).astype(np.float32)

y_mean_x = float(np.mean(y_train_x)); y_std_x = float(np.std(y_train_x))
if (not np.isfinite(y_std_x)) or (y_std_x < 1e-8): y_std_x = 1.0
y_train_x_s = ((y_train_x - y_mean_x) / y_std_x).astype(np.float32)

seed_all(SEED)
t1 = time.time()
model_x = make_model(d_in=X_train_x_s.shape[1])
model_x = train_bnn(model_x, X_train_x_s, y_train_x_s)
t2 = time.time()
train_time_x = round(t2-t1, 2)

rows_x = []
for i in TESTSET:
    test17 = read_csv_safe(DATA_PATH / f"Turbine{i}_2017.csv")
    if test17 is None or not set(FEATURES+[TARGET]).issubset(test17.columns): continue
    test17 = test17[FEATURES+[TARGET]].dropna()
    if test17.empty: continue
    X_te = test17[FEATURES].to_numpy(np.float32)
    y_te = test17[TARGET].to_numpy(np.float32)
    X_te_s = x_scaler_x.transform(X_te).astype(np.float32)
    p1 = time.time()
    pred_s, _ = predict_mc(model_x, X_te_s, mc_draws=MC_DRAWS)
    p2 = time.time()
    pred = np.clip(pred_s * y_std_x + y_mean_x, 0.0, None)
    rows_x.append({"Turbine":i,"Method":"BNN(x)",
                   "RMSE_2017":float(np.sqrt(mean_squared_error(y_te, pred))),
                   "Runtime_train":train_time_x,"Runtime_pred17":round(p2-p1,2)})

results_x_bnn = pd.DataFrame(rows_x)
results_x_bnn.to_csv(OUT_CSV.parent/"bnn_table4_x.csv", index=False)
bnn_x_rmse = results_x_bnn["RMSE_2017"].mean() if not results_x_bnn.empty else np.nan
update_final_results(method="Bayesian NN", table_id="Table 4", rmse=bnn_x_rmse, version="x")
